# Qwen3.5-9B Brand Voice Experiment With llama.cpp

This notebook fine-tunes `Qwen3.5-9B` with `Unsloth` QLoRA, saves the LoRA adapter, saves the merged model, and exports both the untouched baseline model and the fine-tuned model to GGUF for `llama.cpp`.

## Runtime Notes

- Intended for Google Colab or another GPU notebook.
- Upload or mount this repo so the notebook can access the run-1 artifacts under `data/processed/`.
- The downstream comparison loop runs locally on your Mac through `llama-server`; this notebook handles training and GGUF export only.
- Use the same `qwen3-instruct` chat template throughout training and export.


In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except:
        _numpy = "numpy"
        _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0.
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0


In [ ]:
from pathlib import Path
import json
import shutil

import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

import torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

ROOT = Path('/content/Fine-tuning-LLMs-HF-')
if not ROOT.exists():
    ROOT = Path.cwd()

DATA_DIR = ROOT / 'data' / 'processed'
ARTIFACTS_DIR = ROOT / 'artifacts'
GGUF_DIR = ARTIFACTS_DIR / 'gguf'
MERGED_DIR = ARTIFACTS_DIR / 'merged'
LORA_DIR = ROOT / 'qwen35_9b_brand_voice_lora'

TRAIN_PATH = DATA_DIR / 'train_run1_qwen35_9b.jsonl'
EVAL_PATH = DATA_DIR / 'eval_run1_qwen35_9b.jsonl'

MODEL_NAME = 'unsloth/Qwen3.5-4B'
MAX_SEQ_LENGTH = 2048
SEED = 3407
QUANTIZATION_METHOD = 'q4_k_m'
BASELINE_GGUF_PATH = GGUF_DIR / 'qwen35_9b_baseline_q4_k_m.gguf'
FINETUNED_GGUF_PATH = GGUF_DIR / 'qwen35_9b_brand_voice_q4_k_m.gguf'

for directory in (ARTIFACTS_DIR, GGUF_DIR, MERGED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print('Repo root:', ROOT)
print('Train path:', TRAIN_PATH)
print('Eval path:', EVAL_PATH)


In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open('r', encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON at {path}:{line_no}') from exc
            rows.append(row)
    return rows


def normalize_message(value, row_key: str) -> list[dict[str, str]]:
    messages = value
    if not isinstance(messages, list) or not messages:
        raise ValueError(f'Missing or empty messages for {row_key}')

    normalized: list[dict[str, str]] = []
    for msg_idx, msg in enumerate(messages):
        if not isinstance(msg, dict):
            raise ValueError(f'Invalid message at {row_key} index {msg_idx}: expected dict')

        role = msg.get('role')
        content = msg.get('content')
        if role not in {'system', 'user', 'assistant'}:
            raise ValueError(f'Invalid role at {row_key} index {msg_idx}: {role!r}')
        if not isinstance(content, str) or not content.strip():
            raise ValueError(f'Invalid or empty content at {row_key} index {msg_idx}')
        normalized.append({'role': str(role), 'content': content.strip()})

    return normalized


def normalize_rows(rows: list[dict], split_name: str) -> list[dict]:
    normalized: list[dict] = []
    for idx, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(f'Row {idx} in {split_name} must be a JSON object')
        key = f'{split_name}:{idx}'
        messages = normalize_message(row.get('messages'), row_key=key)
        normalized.append({
            'messages': messages,
            'id': row.get('id', ''),
            'platform': row.get('platform', ''),
            'source_type': row.get('source_type', ''),
            'created_at': row.get('created_at', ''),
            'topic': row.get('topic', ''),
            'post_type': row.get('post_type', ''),
            'length_bucket': row.get('length_bucket', ''),
        })
    return normalized


def rows_to_dataset(rows: list[dict], tokenizer) -> Dataset:
    dataset = Dataset.from_list(rows)
    keep_cols = [col for col in dataset.column_names if col != 'messages']

    def format_batch(batch):
        texts = [
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            for messages in batch['messages']
        ]
        return {'text': texts}

    return dataset.map(format_batch, batched=True, remove_columns=keep_cols)


def validate_dataset_schema(rows: list[dict], split_name: str) -> None:
    if not rows:
        raise ValueError(f'{split_name} dataset is empty')
    for idx, row in enumerate(rows):
        messages = row.get('messages')
        if not isinstance(messages, list):
            raise ValueError(f'{split_name}[{idx}] has invalid messages payload')
        if len([m for m in messages if isinstance(m, dict) and m.get('role') == 'user']) == 0:
            raise ValueError(f'{split_name}[{idx}] has no user message')
        if len([m for m in messages if isinstance(m, dict) and m.get('role') == 'assistant']) == 0:
            raise ValueError(f'{split_name}[{idx}] has no assistant message')

    print(f'{split_name} rows:', len(rows))


def rename_single_gguf(export_dir: Path, target_path: Path) -> None:
    candidates = sorted(export_dir.glob('*.gguf'))
    if len(candidates) != 1:
        raise ValueError(f'Expected exactly one GGUF file in {export_dir}, found {len(candidates)}')
    target_path.parent.mkdir(parents=True, exist_ok=True)
    candidates[0].replace(target_path)
    shutil.rmtree(export_dir, ignore_errors=True)


def count_non_empty_jsonl_lines(path: Path) -> int:
    count = 0
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if line.strip():
                count += 1
    return count


In [ ]:
train_rows_raw = read_jsonl(TRAIN_PATH)
eval_rows_raw = read_jsonl(EVAL_PATH)

train_source_lines = count_non_empty_jsonl_lines(TRAIN_PATH)
eval_source_lines = count_non_empty_jsonl_lines(EVAL_PATH)
if len(train_rows_raw) != train_source_lines:
    raise ValueError(f'Train parse mismatch: parsed {len(train_rows_raw)} rows but file has {train_source_lines} non-empty lines')
if len(eval_rows_raw) != eval_source_lines:
    raise ValueError(f'Eval parse mismatch: parsed {len(eval_rows_raw)} rows but file has {eval_source_lines} non-empty lines')

# Normalize and validate schema to keep training robust even with extra metadata.
train_rows = normalize_rows(train_rows_raw, "train")
eval_rows = normalize_rows(eval_rows_raw, "eval")
validate_dataset_schema(train_rows, "train")
validate_dataset_schema(eval_rows, "eval")


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    load_in_16bit=False,
    full_finetuning=False,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template='qwen3-instruct',
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)


In [ ]:
train_dataset = rows_to_dataset(train_rows, tokenizer)
eval_dataset = rows_to_dataset(eval_rows, tokenizer)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=3,
        learning_rate=1e-4,
        warmup_steps=5,
        logging_steps=2,
        eval_strategy='steps',
        eval_steps=10,
        save_strategy='steps',
        save_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        optim='adamw_8bit',
        weight_decay=0.001,
        lr_scheduler_type='linear',
        seed=SEED,
        output_dir=str(ROOT / 'outputs_qwen35_9b'),
        report_to='none',
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user',
    response_part='<|im_start|>assistant',
)



In [ ]:
trainer_stats = trainer.train()
trainer_stats


In [ ]:
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))
print('LoRA saved to', LORA_DIR)


## Export Notes

The cells below export:
- the untouched baseline `Qwen3.5-9B` to GGUF
- the merged fine-tuned model to GGUF

If Colab runs out of memory during direct GGUF export, keep the merged 16-bit save and rerun the GGUF export on a higher-memory runtime.


In [ ]:
torch.cuda.empty_cache()

baseline_model, baseline_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=False,
    load_in_16bit=True,
    full_finetuning=False,
)

baseline_tokenizer = get_chat_template(
    baseline_tokenizer,
    chat_template='qwen3-instruct',
)

baseline_export_dir = GGUF_DIR / 'baseline_tmp'
baseline_export_dir.mkdir(parents=True, exist_ok=True)
baseline_model.save_pretrained_gguf(
    str(baseline_export_dir),
    baseline_tokenizer,
    quantization_method=QUANTIZATION_METHOD,
)
rename_single_gguf(baseline_export_dir, BASELINE_GGUF_PATH)
print('Baseline GGUF:', BASELINE_GGUF_PATH)


In [ ]:
model.save_pretrained_merged(
    str(MERGED_DIR / 'qwen35_9b_brand_voice_merged'),
    tokenizer,
    save_method='merged_16bit',
)
print('Merged model saved to', MERGED_DIR / 'qwen35_9b_brand_voice_merged')


In [ ]:
finetuned_export_dir = GGUF_DIR / 'finetuned_tmp'
finetuned_export_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained_gguf(
    str(finetuned_export_dir),
    tokenizer,
    quantization_method=QUANTIZATION_METHOD,
)
rename_single_gguf(finetuned_export_dir, FINETUNED_GGUF_PATH)
print('Fine-tuned GGUF:', FINETUNED_GGUF_PATH)


## Local Next Step

After the notebook finishes, run the local evaluation suite on your Mac:

```bash
python3 scripts/run_llamacpp_suite.py   --prompts data/processed/eval_suite_run1_social_20x3.jsonl   --baseline-gguf artifacts/gguf/qwen35_9b_baseline_q4_k_m.gguf   --finetuned-gguf artifacts/gguf/qwen35_9b_brand_voice_q4_k_m.gguf
```
